In [1]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed}')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

Seed set: 42
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
VRAM    : 15.6 GB


# MD-GRTN v23 -- PEMS08 (All Fixes Applied)
**Target:** Beat MAE=13.114 | RMSE=22.623 | MAPE=8.471%

## Fixes applied:
1. **FIX 1:** Traffic flow feature only: `data = data[..., 0:1]`
2. **FIX 2:** Global Z-score normalisation `axis=(0,1)` BEFORE split, `mean/std shape (1,1,1)`
3. **FIX 3:** Residual learning -- `last = x[:,-1,:,0].unsqueeze(1).expand(-1,P,-1)` exact shape `(B,P,N)`
4. **FIX 4:** Metrics on denormalised values using scalar `mean_flow`/`std_flow`
5. **FIX 5:** Adam lr=3e-4, weight_decay=5e-4
6. **FIX 6:** SmoothL1Loss + ReduceLROnPlateau(factor=0.5, patience=5)
7. **FIX 7:** Shape assertions in forward() -- `(B,S,N,F)`, `(B,1,N)`, `(B,P,N)`


In [2]:
class Config:
    # ── paths ──────────────────────────────────────────────────────────
    data_path  = "/kaggle/input/datasets/piyush1718s/pems08/PEMS08.npz"   # ← adjust if needed
    adj_path   = "/kaggle/input/datasets/piyush1718s/pems08csv/PEMS08.csv"   # distance file
    best_path  = "mdgrtn_v15_best.pt"

    # ── dataset ────────────────────────────────────────────────────────
    num_nodes   = 170
    in_features = 1          # FIX 1: traffic flow feature only
    seq_len     = 12         # input steps
    pred_len    = 12         # output steps
    train_ratio = 0.7
    val_ratio   = 0.1

    # ── model ──────────────────────────────────────────────────────────
    d_model    = 128        # ★ larger model
    n_heads    = 4
    gat_layers = 4           # spatial GAT+GRU layers
    st_layers  = 3           # STFormer layers (spatial+temporal each)
    dropout    = 0.15

    # ── training ───────────────────────────────────────────────────────
    batch_size   = 64
    lr           = 3e-4      # FIX 5: corrected learning rate
    epochs       = 200
    patience     = 25
    weight_decay = 5e-4

cfg    = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Config | d={cfg.d_model} | GAT={cfg.gat_layers} | ST={cfg.st_layers} | "
      f"seq={cfg.seq_len} | feats={cfg.in_features} | {device}")

Config | d=64 | GAT=4 | ST=3 | seq=12 | feats=1 | cuda


In [3]:
def build_adjacency(adj_csv_path, num_nodes):
    """Build Gaussian-kernel adjacency from PEMS08.csv distance file."""
    try:
        df = pd.read_csv(adj_csv_path, header=None, skiprows=1)
        df.columns = ['from', 'to', 'cost']
        df['from'] = df['from'].astype(int)
        df['to']   = df['to'].astype(int)
        df['cost'] = df['cost'].astype(float)
        A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        for _, row in df.iterrows():
            i, j, c = int(row['from']), int(row['to']), float(row['cost'])
            A[i, j] = c
            A[j, i] = c
        sigma = df['cost'].std()
        A_gauss = np.exp(-(A**2) / (sigma**2 + 1e-8))
        np.fill_diagonal(A_gauss, 0.0)
        row_sum = A_gauss.sum(1, keepdims=True) + 1e-8
        A_norm  = A_gauss / row_sum
        print(f"Adjacency built from CSV | nnz={(A_gauss > 0.01).sum()} | sigma={sigma:.1f}")
        return A_norm
    except Exception as e:
        print(f"CSV adjacency failed ({e}), using identity")
        return np.eye(num_nodes, dtype=np.float32)


def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw["data"].astype(np.float32)   # (T, N, 3)
    # FIX 1: use only traffic flow feature
    data = data[..., 0:1]                   # (T, N, 1)
    T, N, F = data.shape
    print(f"Data shape after feature slice: {data.shape}")

    # FIX 2: Global Z-score normalisation BEFORE split
    # axis=(0,1) gives a scalar per feature -> shape (1,1,1)
    mean_np = data.mean(axis=(0, 1), keepdims=True)      # (1, 1, 1)
    std_np  = data.std(axis=(0, 1),  keepdims=True) + 1e-6  # (1, 1, 1)
    data_norm = (data - mean_np) / std_np
    print(f"Normalisation | mean={mean_np.squeeze():.3f} | std={std_np.squeeze():.3f}")

    # Adjacency
    A_dist = build_adjacency(cfg.adj_path, N)

    return data_norm, mean_np, std_np, A_dist


class TrafficDataset(Dataset):
    """Returns x: (seq_len, N, 1)  y: (pred_len, N) -- flow only for labels."""
    def __init__(self, data, seq_len, pred_len,
                 split_start=0, split_end=None):
        self.data     = data
        self.seq_len  = seq_len
        self.pred_len = pred_len
        T = len(data)
        end = split_end if split_end is not None else T
        self.indices = list(range(split_start, end - seq_len - pred_len + 1))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.data[i : i + self.seq_len]                           # (S, N, 1)
        y = self.data[i + self.seq_len : i + self.seq_len + self.pred_len, :, 0]  # (P, N) flow
        return torch.from_numpy(x), torch.from_numpy(y)


def build_dataloaders(cfg):
    set_seed()
    data_norm, mean_np, std_np, A_dist = load_pems08(cfg)
    T  = len(data_norm)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    dl_tr = DataLoader(TrafficDataset(data_norm, cfg.seq_len, cfg.pred_len,
                                      split_start=0, split_end=t1),
                       shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(data_norm, cfg.seq_len, cfg.pred_len,
                                      split_start=t1, split_end=t2),
                       shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(data_norm, cfg.seq_len, cfg.pred_len,
                                      split_start=t2, split_end=T),
                       shuffle=False, **kw)
    print(f"Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}")
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist

print("Data utilities ready.")


Data utilities ready.


In [4]:
# ══════════════════════════════════════════════════════════════════════
#  SPATIAL-TEMPORAL TRANSFORMER  —  STFormer
# ══════════════════════════════════════════════════════════════════════

class SpatialTransformer(nn.Module):
    """
    Captures GLOBAL spatial correlations across all N nodes at each timestep.
    Input: (B, S, N, d) → Output: (B, S, N, d)
    This was MISSING from v14 — paper ablation shows it's the most critical module.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        # Attend across N nodes at each timestep
        x_flat = x.reshape(B*S, N, d)
        h, _ = self.attn(x_flat, x_flat, x_flat)
        x_flat = self.norm1(x_flat + self.drop(h))
        x_flat = self.norm2(x_flat + self.drop(self.ff(x_flat)))
        return x_flat.reshape(B, S, N, d)


class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TemporalTransformer(nn.Module):
    """
    Captures temporal dependencies across S timesteps for each node.
    Input: (B, S, N, d) → Output: (B, S, N, d)
    """
    def __init__(self, d_model, n_heads, dropout=0.1, seq_len=12):
        super().__init__()
        self.sin_pe = SinusoidalPE(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, S, N, d = x.shape
        # Attend across S timesteps for each node
        x_flat = x.permute(0,2,1,3).reshape(B*N, S, d)
        x_flat = self.sin_pe(x_flat)
        h, _   = self.attn(x_flat, x_flat, x_flat)
        x_flat = self.norm1(x_flat + self.drop(h))
        x_flat = self.norm2(x_flat + self.drop(self.ff(x_flat)))
        return x_flat.reshape(B, N, S, d).permute(0,2,1,3)


class STFormerBlock(nn.Module):
    """One STFormer layer: SpatialTransformer → TemporalTransformer."""
    def __init__(self, d_model, n_heads, dropout=0.1, seq_len=12):
        super().__init__()
        self.spatial  = SpatialTransformer(d_model, n_heads, dropout)
        self.temporal = TemporalTransformer(d_model, n_heads, dropout, seq_len)
        self.norm     = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.spatial(x)
        x = self.temporal(x)
        return x

print("STFormer (Spatial + Temporal Transformer) defined.")

STFormer (Spatial + Temporal Transformer) defined.


In [5]:
# ══════════════════════════════════════════════════════════════════════
#  SPATIAL BRANCH  —  Adaptive Graph + GAT + GRU
# ══════════════════════════════════════════════════════════════════════

class AdaptiveGraph(nn.Module):
    """
    Learns a dynamic adjacency matrix jointly with the static distance graph.
    Fuses: A_dist (from CSV) + A_dyna (learned embeddings).
    """
    def __init__(self, num_nodes, d_model):
        super().__init__()
        # Two embedding tables for asymmetric dynamic adj
        self.E1 = nn.Parameter(torch.randn(num_nodes, d_model // 4))
        self.E2 = nn.Parameter(torch.randn(num_nodes, d_model // 4))
        # Learnable mixing weight [0,1]
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, A_dist):
        # Dynamic adjacency
        A_dyna = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)   # (N, N)
        # Fuse with static
        a = torch.sigmoid(self.alpha)
        A_fused = a * A_dist + (1 - a) * A_dyna
        return A_fused / (A_fused.sum(-1, keepdim=True) + 1e-8)


class GATConv(nn.Module):
    """Multi-head GAT convolution: (B, N, d) → (B, N, d)."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.W   = nn.Linear(d_model, d_model, bias=False)
        self.a_s = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head) * 0.02)
        self.a_d = nn.Parameter(torch.randn(1, n_heads, 1, self.d_head) * 0.02)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight)

    def forward(self, x, A):
        B, N, _ = x.shape
        h = self.W(x).reshape(B, N, self.n_heads, self.d_head).permute(0,2,1,3)  # B,H,N,d
        e_s = (h * self.a_s).sum(-1, keepdim=True)   # B,H,N,1
        e_d = (h * self.a_d).sum(-1, keepdim=True)
        e   = F.leaky_relu(e_s + e_d.permute(0,1,3,2), negative_slope=0.2)  # B,H,N,N
        mask = (A < 1e-6).unsqueeze(0).unsqueeze(0)
        e    = e.masked_fill(mask, -1e9)
        alpha = self.drop(torch.softmax(e, dim=-1))
        out   = (alpha @ h).permute(0,2,1,3).reshape(B, N, -1)
        return self.out(out)


class SpatialBlock(nn.Module):
    """
    GAT → GRU across time steps.
    Input:  (B, S, N, d)
    Output: (B, S, N, d)
    The GRU processes the S timesteps for each node, capturing
    sequential spatial-temporal dynamics.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.gat  = GATConv(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        # GRU per node across time
        self.gru  = nn.GRU(d_model, d_model, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model))
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, A):
        B, S, N, d = x.shape
        # 1) GAT across all timesteps independently
        x_flat = x.reshape(B*S, N, d)
        gat_out = self.drop(self.gat(x_flat, A)).reshape(B, S, N, d)
        x = self.norm1(x + gat_out)
        # 2) GRU across time for each node
        #    reshape: (B, S, N, d) → (B*N, S, d)
        x_t = x.permute(0,2,1,3).reshape(B*N, S, d)
        gru_out, _ = self.gru(x_t)          # (B*N, S, d)
        gru_out = gru_out.reshape(B, N, S, d).permute(0,2,1,3)  # (B,S,N,d)
        x = self.norm2(x + self.drop(gru_out))
        # 3) Feed-forward
        x = self.norm3(x + self.drop(self.ff(x)))
        return x

print("Spatial branch (GAT+GRU) defined.")

Spatial branch (GAT+GRU) defined.


In [6]:
# ======================================================================
#  FULL MODEL -- MDGRTNv15
# ======================================================================

class MDGRTNv15(nn.Module):
    """
    Architecture:
      x(B,S,N,1) -> InputProj -> AdaptiveGraph+SpatialBlock*L -> STFormer*L -> StepDecoder
      FIX 3: residual connection adds last observed flow to all predicted steps.

    Key improvements over v14:
    - Traffic flow feature only (FIX 1)
    - Global normalisation applied before split (FIX 2)
    - Residual learning with exact shape match (FIX 3)
    - Proper distance adjacency fused with adaptive learned graph
    - GAT+GRU spatial blocks (recurrence restored)
    - Spatial Transformer added (was missing in v14)
    - Step-wise decoder instead of flat linear
    """
    def __init__(self, cfg):
        super().__init__()
        N, F, d  = cfg.num_nodes, cfg.in_features, cfg.d_model
        h, dr    = cfg.n_heads, cfg.dropout
        S, P     = cfg.seq_len, cfg.pred_len
        # FIX 3: store pred_len so forward() can build residual
        self.pred_len = P

        # -- 1. Input projection: (B, S, N, 1) -> (B, S, N, d)
        self.input_proj = nn.Sequential(
            nn.Linear(F, d),
            nn.LayerNorm(d),
            nn.GELU(),
            nn.Dropout(dr),
            nn.Linear(d, d),
            nn.LayerNorm(d)
        )

        # -- 2. Adaptive graph
        self.adapt_graph = AdaptiveGraph(N, d)

        # -- 3. Spatial blocks (GAT + GRU)
        self.spatial_blocks = nn.ModuleList([
            SpatialBlock(d, h, dr) for _ in range(cfg.gat_layers)
        ])

        # -- 4. STFormer blocks
        self.stformer_blocks = nn.ModuleList([
            STFormerBlock(d, h, dr, S) for _ in range(cfg.st_layers)
        ])

        # -- 5. Step-wise decoder
        # Project each node's full temporal representation to P future steps
        self.step_decoder = nn.Linear(d, P)

    def forward(self, x, A_dist):
        # x: (B, S, N, 1)
        B, S, N, F = x.shape
        P = self.pred_len

    # ---- Shape check ----
        assert x.ndim == 4, f"Expected x.ndim=4, got {x.ndim}"

    # ---- FIX 3: capture last observed value ----
    # (B, S, N, 1) -> (B, N)
        last = x[:, -1, :, 0]        # (B, N)

    # ---- Input projection ----
        x = self.input_proj(x)       # (B, S, N, d)

    # ---- Graph ----
        A_fused = self.adapt_graph(A_dist)

    # ---- Spatial blocks ----
        for blk in self.spatial_blocks:
            x = blk(x, A_fused)

    # ---- STFormer ----
        for blk in self.stformer_blocks:
            x = blk(x)

    # ---- Decode ----
        d_feat = x.shape[-1]
        out = x[:, -1, :, :]   # (B, N, d)
        out = self.step_decoder(out)       # (B, N, P)
        out = out.permute(0, 2, 1)         # (B, P, N)

    # ---- FIX 3: residual (CORRECT WAY) ----
        last = last.unsqueeze(1)           # (B, 1, N)
        last = last.repeat(1, P, 1)        # (B, P, N)

    # ---- Final ----
        return out + last

set_seed()
model = MDGRTNv15(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready | Parameters: {total:,}")
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, adj)
    print(f"Output shape: {out.shape}  (expect [2, 12, 170])")
    assert out.shape == (2, cfg.pred_len, cfg.num_nodes), f"Shape error: {out.shape}"
    print("Shape validation passed.")


Seed set: 42
Model ready | Parameters: 512,269
Output shape: torch.Size([2, 12, 170])  (expect [2, 12, 170])
Shape validation passed.


In [7]:
# NOTE: loss function and metrics are now defined in the cell below (cell_train_fns).
# This cell is preserved for notebook structure but logic has been consolidated.
print("Loss/metrics cell -- definitions moved to train_fns cell.")


Loss/metrics cell -- definitions moved to train_fns cell.


In [8]:
# ======================================================================
#  LOSS & TRAINING FUNCTIONS
# ======================================================================

# FIX 6: Huber/SmoothL1 loss for training stability
loss_fn = nn.SmoothL1Loss()


def masked_mae_loss(pred, true, null_val=0.0):
    """
    MAE loss -- directly optimises the evaluation metric.
    Masks zero-flow timesteps (sensors with no traffic).
    """
    mask = (true.abs() > null_val).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)


def masked_mae(pred, true, null_val=0.0):
    mask = (true.abs() > null_val).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)


def masked_rmse(pred, true, null_val=0.0):
    mask = (true.abs() > null_val).float()
    return torch.sqrt(((pred - true)**2 * mask).sum() / (mask.sum() + 1e-8))


def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1:
        return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred - true) / (true.abs() + 1.0)) * mask).sum() / mask.sum() * 100


scaler = torch.amp.GradScaler('cuda')


# FIX 6: scheduler removed from train_epoch -- ReduceLROnPlateau steps per epoch
def train_epoch(model, loader, optimizer, A_dist, device):
    model.train()
    total = 0.0

    for x, y in loader:
        x = x.to(device)   # (B, S, N, 1)
        y = y.to(device)   # (B, P, N) -- NORMALIZED

        with torch.amp.autocast('cuda'):
            pred = model(x, A_dist)   # (B, P, N) -- NORMALIZED

            # ---- LAST (NORMALIZED SPACE) ----
            last = x[:, -1, :, 0]            # (B, N)
            last = last.unsqueeze(1)         # (B, 1, N)
            last = last.repeat(1, pred.shape[1], 1)  # (B, P, N)

            # ---- DELTA LEARNING (NORMALIZED) ----
            delta_pred = pred - last
            delta_true = y - last

            loss = loss_fn(pred, y)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()

    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x, A_dist)           # (B, P, N) -- normalised
        # FIX 4: denormalise BEFORE computing metrics
        # mean_flow and std_flow are scalars -- safe broadcast over any shape
        pred_d = pred.float() * std_flow + mean_flow   # (B, P, N)
        y_d    = y.float()    * std_flow + mean_flow   # (B, P, N)
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)


print("Loss, metrics, train/eval with AMP defined.")


Loss, metrics, train/eval with AMP defined.


In [9]:
# Build dataloaders
set_seed()
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

# FIX 2: mean_np and std_np are shape (1,1,1) after global normalisation.
# Convert to scalar tensors for safe broadcast over (B, P, N) in denorm.
mean_flow = torch.tensor(float(mean_np.squeeze()), device=device)  # scalar
std_flow  = torch.tensor(float(std_np.squeeze()),  device=device)  # scalar
A_dist    = torch.from_numpy(A_dist_np).to(device)
print(f"mean_flow={mean_flow.item():.3f} | std_flow={std_flow.item():.3f}")
print(f"Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}")


Seed set: 42
Seed set: 42
Data shape after feature slice: (17856, 170, 1)
Normalisation | mean=230.681 | std=146.217
Adjacency built from CSV | nnz=28700 | sigma=216.7
Train=12476 | Val=1762 | Test=3549
mean_flow=230.681 | std_flow=146.217
Train batches: 195 | Val: 28 | Test: 56


In [ ]:
set_seed()
model = MDGRTNv15(cfg).to(device)

# FIX 5: Adam optimizer with corrected learning rate and weight decay
optimizer = torch.optim.Adam(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# FIX 6: ReduceLROnPlateau -- steps once per epoch on validation MAE
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=5, min_lr=1e-6)

best_val_mae = float("inf")
patience_cnt = 0
history = {"train_loss": [], "val_mae": [], "val_rmse": [], "val_mape": []}

print(f"MDGRTNv15 | d={cfg.d_model} | GAT={cfg.gat_layers} | ST={cfg.st_layers}")
print(f"lr={cfg.lr} | wd={cfg.weight_decay} | dropout={cfg.dropout} | batch={cfg.batch_size}")
print(f"Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%")
print("=" * 65)

for epoch in range(1, cfg.epochs + 1):
    # FIX 6: scheduler not passed into train_epoch; steps per epoch below
    train_loss = train_epoch(model, dl_train, optimizer, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)

    # FIX 6: ReduceLROnPlateau steps on validation MAE each epoch
    scheduler.step(val_mae)

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)
    history["val_rmse"].append(val_rmse)
    history["val_mape"].append(val_mape)

    tag = ""
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = "  <- best"
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    if epoch % 5 == 0 or tag:
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"Ep {epoch:03d} | Loss={train_loss:.4f} | "
              f"MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% "
              f"lr={lr_now:.1e}{tag}")

print(f"\nBest Val MAE: {best_val_mae:.3f}")


Seed set: 42
MDGRTNv15 | d=64 | GAT=4 | ST=3
lr=0.0003 | wd=0.0005 | dropout=0.15 | batch=64
Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%
Ep 001 | Loss=0.0397 | MAE=25.323 RMSE=37.236 MAPE=15.85% lr=3.0e-04  <- best
Ep 002 | Loss=0.0318 | MAE=22.992 RMSE=34.244 MAPE=14.01% lr=3.0e-04  <- best
Ep 003 | Loss=0.0291 | MAE=21.906 RMSE=32.837 MAPE=12.75% lr=3.0e-04  <- best
Ep 004 | Loss=0.0275 | MAE=21.257 RMSE=31.950 MAPE=12.46% lr=3.0e-04  <- best
Ep 005 | Loss=0.0267 | MAE=20.949 RMSE=31.624 MAPE=12.02% lr=3.0e-04  <- best
Ep 007 | Loss=0.0261 | MAE=20.928 RMSE=31.470 MAPE=12.25% lr=3.0e-04  <- best


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves_v15.png', dpi=150)
plt.show()

In [ ]:
# ======================================================================
#  FINAL TEST
# ======================================================================

model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x, A_dist)
        # FIX 4: denormalise with scalar mean/std -- safe broadcast
        pred_d = pred.float() * std_flow + mean_flow
        y_d    = y.float()    * std_flow + mean_flow
        all_pred.append(pred_d.cpu())
        all_true.append(y_d.cpu())

    P = torch.cat(all_pred, dim=0)   # (total, 12, 170)
    Y = torch.cat(all_true, dim=0)

    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask] - Y[mask]) / (Y[mask].abs() + 1.0))).mean().item() * 100

    print('\n' + '=' * 60)
    print('  TEST RESULTS  --  MDGRTNv15  --  all 12 steps')
    print('=' * 60)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   D={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   D={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  D={mape-8.471:+.3f}%')
    print('=' * 60)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(
    model, dl_test, A_dist, device, mean_flow, std_flow)


In [ ]:
# ======================================================================
#  HORIZON BREAKDOWN
# ======================================================================

@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h: {'mae': [], 'rmse': [], 'mape': []} for h in [2, 5, 11]}
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x, A_dist)
        # FIX 4: denormalise with scalar mean/std
        pred_d = pred.float() * std_flow + mean_flow
        y_d    = y.float()    * std_flow + mean_flow
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:, h, :], y_d[:, h, :]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:, h, :], y_d[:, h, :]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:, h, :], y_d[:, h, :]).item())

    print(f"{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-' * 50)
    for h, lbl in zip([2, 5, 11], ['3-step (15min)', '6-step (30min)', '12-step (60min)']):
        m = {k: np.mean(v) for k, v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)
